In [ ]:
!pip install -U vllm requests soundfile librosa pyopenjtalk g2p-en
!pip install -U git+https://github.com/Aratako/MioCodec.git
!git clone https://github.com/Aratako/MioTTS-Inference.git

In [ ]:
import subprocess, time, requests, os

# kill old servers
!pkill -f "vllm serve" || true

# start vLLM
p = subprocess.Popen(
    "vllm serve beleata74/mio-tts-0.6b-bg-finetuned "
    "--port 8000 --gpu-memory-utilization 0.6 "
    "> /tmp/vllm.log 2>&1",
    shell=True,
)

for i in range(120):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=3)
        print("vLLM up", r.status_code)
        break
    except Exception as e:
        if i % 10 == 0:
            print(f"waiting... {i*2}s")
            !tail -n 40 /tmp/vllm.log
        time.sleep(2)
else:
    !tail -n 120 /tmp/vllm.log
    raise RuntimeError("vLLM did not start")

In [3]:
%cd /content/MioTTS-Inference
import subprocess, time, requests

subprocess.Popen(
    "python run_server.py --llm-base-url http://127.0.0.1:8000/v1 > /tmp/miotts.log 2>&1 &",
    shell=True,
)

for _ in range(120):
    try:
        r = requests.get("http://127.0.0.1:8001/health", timeout=3)
        print("MioTTS server up", r.status_code, r.text)
        break
    except:
        time.sleep(2)
else:
    raise RuntimeError("MioTTS server did not start")

/content/MioTTS-Inference
MioTTS server up 200 {"status":"ok"}


In [20]:
from google.colab import files
uploaded = files.upload()
reference_path = next(iter(uploaded.keys()))
print(reference_path)

Saving bg-test-mp.mp3 to bg-test-mp.mp3
bg-test-mp.mp3


In [28]:
import requests
from IPython.display import Audio

text = "Ало, здравейте… Днес се обаждам във връзка с едно съвсем дребно, но малко необичайно уточнение при вас, което по някаква причина е останало в системата като висящо и сега излиза, че трябва да го затворим ръчно, защото иначе продължава "

with open(reference_path, "rb") as f:
    r = requests.post(
        "http://127.0.0.1:8001/v1/tts/file",
        files={"reference_audio": (reference_path, f, "audio/wav")},
        data={"text": text},
        timeout=180,
    )

print(r.status_code)
open("/content/big-message.wav", "wb").write(r.content)
Audio("/content/big-message.wav")

200


In [27]:
import time, requests, soundfile as sf

texts = [
    "Ало! ЗДРАВЕЙТЕ!!! Това е СПЕШНО!!!",
    "АМиииии мислих си за теб доста, все пак ....... ъъъъ .... всъщност, да ти кажа, честно не знам",
]

t0 = time.perf_counter()

for i, text in enumerate(texts, 1):
    with open(reference_path, "rb") as f:
        r = requests.post(
            "http://127.0.0.1:8001/v1/tts/file",
            files={"reference_audio": (reference_path, f, "audio/wav")},
            data={"text": text},
            timeout=180,
        )
    open(f"/content/out_{i}.wav", "wb").write(r.content)

wall = time.perf_counter() - t0
audio_sec = 0.0
for i in range(1, len(texts)+1):
    audio, sr = sf.read(f"/content/out_{i}.wav")
    audio_sec += len(audio) / sr

print("wall_sec =", round(wall, 2))
print("audio_sec =", round(audio_sec, 2))
print("rtf =", round(wall / audio_sec, 3))
print("est_modal_t4_cost_usd =", round(wall * 0.000164, 5))

wall_sec = 3.07
audio_sec = 11.16
rtf = 0.275
est_modal_t4_cost_usd = 0.0005


In [23]:
source_path = "bg-test-mp.mp3"   # change this
wav_path = "bg-test-wav.wav"

#!ffmpeg -y -i "{source_path}" -ar 24000 -ac 1 "{wav_path}"

reference_path = wav_path
print("Converted to:", reference_path)

Converted to: bg-test-wav.wav


In [ ]:
!pip install librosa soundfile scipy

In [30]:
import numpy as np
import librosa
import soundfile as sf
from scipy.signal import butter, sosfilt
from IPython.display import Audio


def bandpass(y, sr, low=90, high=7200):
    sos = butter(4, [low, high], btype="bandpass", fs=sr, output="sos")
    return sosfilt(sos, y)


def high_shelf_cut(y, sr, cutoff=4200, amount=0.35):
    fft = np.fft.rfft(y)
    freqs = np.fft.rfftfreq(len(y), 1 / sr)
    fft[freqs > cutoff] *= amount
    return np.fft.irfft(fft, n=len(y))


def soft_saturation(y, drive=1.8):
    return np.tanh(y * drive) / np.tanh(drive)


def add_breath_noise(y, sr, amount=0.004):
    noise = np.random.normal(0, amount, len(y))
    noise = bandpass(noise, sr, low=2500, high=9000)
    return y + noise


def normalize(y, peak=0.95):
    m = np.max(np.abs(y))
    return y if m == 0 else y / m * peak


def transform_voice(
    input_path,
    output_path,
    pitch_steps=-1.5,
    speed=0.94,
    saturation=1.6,
    noise=0.004,
):
    y, sr = librosa.load(input_path, sr=None, mono=True)

    y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch_steps)
    y = librosa.effects.time_stretch(y, rate=speed)

    y = high_shelf_cut(y, sr, cutoff=3800, amount=0.45)
    y = soft_saturation(y, drive=saturation)
    y = add_breath_noise(y, sr, amount=noise)

    y = bandpass(y, sr, low=80, high=7600)
    y = normalize(y)

    sf.write(output_path, y, sr)
    return output_path, sr

In [35]:
out_path, sr = transform_voice(
    "/content/big-message.wav",
    "/content/grandma_rusty.wav",
    pitch_steps=+10.0,   # ↑ pitch → more feminine
    speed=1.28,         # ↑ speed → nervous energy
    saturation=1.4,     # keep low (don’t make it rough)
    noise=0.001         # slight instability
)

Audio(out_path)